# Drone Object Detection — YOLOv8 Training

Edit **Cell 1** to select a training mode, then run all cells top to bottom.

| Mode | Model | Dataset | Notes |
|------|-------|---------|-------|
| `nano` | yolov8n | combined | Fast baseline |
| `medium` | yolov8m | combined | Higher capacity |
| `tuned` | yolov8n | visdrone | Nano with augmentation tuning |

In [ ]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────────
# This is the only cell you need to edit before running the notebook.

MODE = "nano"  # "nano" | "medium" | "tuned"

DRIVE_BASE = "/content/drive/MyDrive/drone_detection_training"

DATASET_ZIP_NAME = {
    "nano":   "combined_yolo.zip",
    "medium": "combined_yolo.zip",
    "tuned":  "visdrone_yolo.zip",
}

MODEL_CONFIG = {
    "nano": {
        "model": "yolov8n.pt",
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
    },
    "medium": {
        "model": "yolov8m.pt",
        "epochs": 50,
        "imgsz": 640,
        "batch": 8,
    },
    "tuned": {
        "model": "yolov8n.pt",
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
        "lr0": 0.001,
        "lrf": 0.01,
        "mosaic": 1.0,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "flipud": 0.1,
        "fliplr": 0.5,
    },
}

assert MODE in MODEL_CONFIG, f"Unknown MODE '{MODE}'. Choose: nano, medium, tuned"
print(f"Mode: {MODE}  |  Model: {MODEL_CONFIG[MODE]['model']}  |  Zip: {DATASET_ZIP_NAME[MODE]}")

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────────
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics>=8.0.0", "torch", "torchvision"],
    check=True
)

import torch

if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {device}  |  VRAM: {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

base = Path(DRIVE_BASE)
runs_dir = base / "runs"
datasets_dir = base / "datasets"

runs_dir.mkdir(parents=True, exist_ok=True)
datasets_dir.mkdir(parents=True, exist_ok=True)

print("Drive folder structure:")
print(f"  {base}/")
print(f"  ├── runs/      {'(exists)' if runs_dir.exists() else '(created)'}")
print(f"  └── datasets/  {'(exists)' if datasets_dir.exists() else '(created)'}")

In [ ]:
# ── Cell 4: Dataset Integrity Check ───────────────────────────────────────────
import shutil, sys
from pathlib import Path

DATASET_DIR = Path("/content/dataset")
zip_name = DATASET_ZIP_NAME[MODE]
zip_path = Path(DRIVE_BASE) / "datasets" / zip_name

# Locate zip
if not zip_path.exists():
    print("ERROR: Dataset zip not found.")
    print()
    print("Please upload the zip to Google Drive at:")
    print(f"  {zip_path}")
    print()
    print("To create the zip locally, run:")
    print(f"  python scripts/zip_for_colab.py --dataset {zip_name.replace('_yolo.zip', '')}")
    print("Then upload colab_uploads/" + zip_name + " to your Drive at the path above.")
    raise SystemExit(1)

# Unzip
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

print(f"Unzipping {zip_name} to {DATASET_DIR} ...")
shutil.unpack_archive(str(zip_path), str(DATASET_DIR))
print("Unzip complete.")

# Verify structure
errors = []
counts = {}

for split in ("train", "val", "test"):
    img_dir = DATASET_DIR / "images" / split
    lbl_dir = DATASET_DIR / "labels" / split

    if not img_dir.exists():
        errors.append(f"Missing: images/{split}/")
        continue
    if not lbl_dir.exists():
        errors.append(f"Missing: labels/{split}/")
        continue

    imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))
    lbls = list(lbl_dir.glob("*.txt"))
    counts[split] = {"images": len(imgs), "labels": len(lbls)}

    if len(imgs) != len(lbls):
        errors.append(f"{split}: {len(imgs)} images vs {len(lbls)} labels (mismatch)")

# Find yaml
yaml_files = list(DATASET_DIR.glob("*.yaml"))
if not yaml_files:
    errors.append("No .yaml file found in dataset root")
else:
    yaml_file = yaml_files[0]

# Print summary
print()
print(f"{'Split':<8} {'Images':>8} {'Labels':>8} {'Match':>7}")
print("-" * 35)
for split, c in counts.items():
    match = "OK" if c["images"] == c["labels"] else "MISMATCH"
    print(f"{split:<8} {c['images']:>8} {c['labels']:>8} {match:>7}")

if yaml_files:
    print(f"\nDataset yaml: {yaml_file.name}")

if errors:
    print()
    print("ERRORS:")
    for e in errors:
        print(f"  - {e}")
    raise SystemExit(1)

print("\nDataset integrity check passed.")

In [ ]:
# ── Cell 5: Resume Check ──────────────────────────────────────────────────────
from datetime import datetime
from pathlib import Path

RESUME = False
base_run_name = f"{MODE}_{datetime.now().strftime('%Y%m%d')}"
run_name = base_run_name

drive_runs = Path(DRIVE_BASE) / "runs"
existing_run = drive_runs / run_name
last_weights = existing_run / "weights" / "last.pt"

if existing_run.exists() and last_weights.exists():
    answer = input(f"Resume existing run '{run_name}'? (yes/no): ").strip().lower()
    if answer in ("yes", "y"):
        RESUME = True
        print(f"Resuming run: {run_name}")
    else:
        # Find a unique run name
        suffix = 2
        while (drive_runs / run_name).exists():
            run_name = f"{base_run_name}_{suffix}"
            suffix += 1
        print(f"Starting new run: {run_name}")
elif existing_run.exists():
    # Run directory exists but no last.pt — start fresh with unique name
    suffix = 2
    while (drive_runs / run_name).exists():
        run_name = f"{base_run_name}_{suffix}"
        suffix += 1
    print(f"Starting new run: {run_name}")
else:
    print(f"Starting new run: {run_name}")

print(f"RESUME = {RESUME}")

In [ ]:
# ── Cell 6: Training ──────────────────────────────────────────────────────────
from pathlib import Path
from ultralytics import YOLO

config = MODEL_CONFIG[MODE].copy()

# Resolve dataset yaml (rewritten by zip_for_colab.py to point at /content/dataset)
zip_stem = DATASET_ZIP_NAME[MODE].replace(".zip", "")  # e.g. combined_yolo
dataset_name = zip_stem.replace("_yolo", "")           # e.g. combined
yaml_path = str(Path("/content/dataset") / f"{dataset_name}.yaml")

model_name = config.pop("model")

print("Training configuration:")
print(f"  Mode:    {MODE}")
print(f"  Model:   {model_name}")
print(f"  Dataset: {yaml_path}")
print(f"  Run:     {run_name}")
print(f"  Resume:  {RESUME}")
for k, v in config.items():
    print(f"  {k}: {v}")
print()

model = YOLO(model_name)

train_kwargs = dict(
    data=yaml_path,
    project="/content/runs",
    name=run_name,
    exist_ok=RESUME,
    resume=RESUME,
    **config,
)

results = model.train(**train_kwargs)

In [ ]:
# ── Cell 7: Save Results to Drive ─────────────────────────────────────────────
import shutil
from pathlib import Path

src_run = Path("/content/runs") / run_name
dst_run = Path(DRIVE_BASE) / "runs" / run_name

if not src_run.exists():
    print(f"ERROR: Run directory not found: {src_run}")
    raise SystemExit(1)

if dst_run.exists():
    shutil.rmtree(dst_run)

shutil.copytree(str(src_run), str(dst_run))

saved_files = sorted(dst_run.rglob("*"))
file_list = [str(f.relative_to(dst_run)) for f in saved_files if f.is_file()]

print(f"Saved {len(file_list)} files to Drive:")
print(f"  {dst_run}")
print()
for name in file_list:
    print(f"  {name}")

In [ ]:
# ── Cell 8: Results Summary ───────────────────────────────────────────────────
import csv
from pathlib import Path

results_csv = Path(DRIVE_BASE) / "runs" / run_name / "results.csv"

if not results_csv.exists():
    print(f"results.csv not found at {results_csv}")
    raise SystemExit(1)

with open(results_csv, newline="") as f:
    rows = list(csv.DictReader(f))

if not rows:
    print("results.csv is empty — training may not have completed.")
    raise SystemExit(1)

# Strip whitespace from keys (ultralytics adds leading spaces)
rows = [{k.strip(): v.strip() for k, v in row.items()} for row in rows]

# Find best epoch by mAP50
MAP50_KEY = "metrics/mAP50(B)"
best_row = max(rows, key=lambda r: float(r.get(MAP50_KEY, 0) or 0))
final_row = rows[-1]
best_epoch = int(float(best_row.get("epoch", len(rows))))

def fmt(row, key):
    v = row.get(key, "N/A")
    try:
        return f"{float(v):.4f}"
    except (ValueError, TypeError):
        return str(v)

print(f"Training complete — {len(rows)} epochs")
print(f"Best epoch: {best_epoch + 1}")
print()
print("Final epoch metrics:")
print(f"  mAP50:      {fmt(final_row, 'metrics/mAP50(B)')}")
print(f"  mAP50-95:   {fmt(final_row, 'metrics/mAP50-95(B)')}")
print(f"  Precision:  {fmt(final_row, 'metrics/precision(B)')}")
print(f"  Recall:     {fmt(final_row, 'metrics/recall(B)')}")
print(f"  Box loss:   {fmt(final_row, 'val/box_loss')}")
print()
print("Best epoch metrics:")
print(f"  mAP50:      {fmt(best_row, 'metrics/mAP50(B)')}")
print(f"  mAP50-95:   {fmt(best_row, 'metrics/mAP50-95(B)')}")
print()
drive_run = Path(DRIVE_BASE) / 'runs' / run_name
print(f"Weights:  {drive_run}/weights/")
print(f"Plots:    {drive_run}/")
print(f"Metrics:  {drive_run}/results.csv")